# Tree of Thoughts — Qwen3.5-2B on Colab

Replicates Table 2 of Yao et al. (NeurIPS 2023) on Game of 24 using Qwen3.5-2B.
Covers all conditions: IO, IO best-of-100, CoT, CoT-SC, CoT best-of-100, ToT (b=1), ToT (b=5).

**Before running:** Runtime → Change runtime type → **T4 GPU**

**GPU usage:** Qwen3.5-2B in bfloat16 uses ~4 GB VRAM — fits entirely on T4 with no CPU offloading.

In [ ]:
import subprocess
print(subprocess.check_output('nvidia-smi', shell=True).decode())

In [ ]:
%pip install -q git+https://github.com/huggingface/transformers.git accelerate>=0.26 sympy pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/DL Final Project/tree-of-thought"

import os
assert os.path.exists('code/run.py'), "run.py not found — check the path above"
assert os.path.exists('data/24.csv'), "data/24.csv not found — check the path above"
print('Paths OK')

In [ ]:
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/DL Final Project/tree-of-thought/.hf_cache"

import sys
sys.path.insert(0, 'code')

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from tot.models import gpt
MODEL = 'hf:Qwen/Qwen3.5-2B'
out = gpt('What is 2 + 2?', model=MODEL, temperature=0.0, max_tokens=32, n=1)
print('Model OK:', out)

---
## Quick Test — IO n=1 (1 puzzle, verbose)
Runs a single puzzle with verbose output so you can inspect the raw LM response.

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --naive_run \
    --prompt_sample standard \
    --n_generate_sample 1 \
    --task_start_index 900 \
    --task_end_index 901 \
    --temperature 0.7 \
    --verbose

## Quick Test — ToT b=5 (1 puzzle, verbose)
Runs a single puzzle with the full ToT BFS so you can inspect the tree search steps and LM responses.

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --method_generate propose \
    --method_evaluate value \
    --method_select greedy \
    --n_generate_sample 1 \
    --n_evaluate_sample 3 \
    --n_select_sample 5 \
    --task_start_index 900 \
    --task_end_index 901 \
    --temperature 0.7 \
    --verbose

---
## Experiments

All conditions from Table 2 of the paper.
Set `START` / `END` to control how many puzzles to run.
The paper evaluates indices 900–999 (100 puzzles).

In [ ]:
START = 900
END   = 1000

### 1. IO Prompt (n=1)
Single direct prompt, no intermediate steps. Paper reports **7.3%** (`accuracy_any`) with GPT-4.

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --naive_run \
    --prompt_sample standard \
    --n_generate_sample 1 \
    --task_start_index {START} \
    --task_end_index {END} \
    --temperature 0.7

### 2. IO Best of 100
100 independent IO samples — oracle metric. Paper reports **33%** (`accuracy_any`) with GPT-4.

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --naive_run \
    --prompt_sample standard \
    --n_generate_sample 100 \
    --task_start_index {START} \
    --task_end_index {END} \
    --temperature 0.7

### 3. CoT Prompt (n=1)
Single chain-of-thought sample. Paper reports **4.0%** (`accuracy_any`) with GPT-4.

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --naive_run \
    --prompt_sample cot \
    --n_generate_sample 1 \
    --task_start_index {START} \
    --task_end_index {END} \
    --temperature 0.7

### 4. CoT-SC (k=100)
100 CoT samples per puzzle.
- `accuracy_avg` = CoT-SC = **9.0%** in the paper
- `accuracy_any` = CoT best of 100 = **49%** in the paper

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --naive_run \
    --prompt_sample cot \
    --n_generate_sample 100 \
    --task_start_index {START} \
    --task_end_index {END} \
    --temperature 0.7

### 5. ToT BFS (b=1)
Propose + LLM value evaluation + greedy selection, beam width 1. Paper reports **45%** (`accuracy_any`) with GPT-4.

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --method_generate propose \
    --method_evaluate value \
    --method_select greedy \
    --n_generate_sample 1 \
    --n_evaluate_sample 3 \
    --n_select_sample 1 \
    --task_start_index {START} \
    --task_end_index {END} \
    --temperature 0.7 \
    --verbose

### 6. ToT BFS (b=5)
Propose + LLM value evaluation + greedy selection, beam width 5. Paper reports **74%** (`accuracy_any`) with GPT-4.

In [ ]:
!python code/run.py \
    --task game24 \
    --backend {MODEL} \
    --method_generate propose \
    --method_evaluate value \
    --method_select greedy \
    --n_generate_sample 1 \
    --n_evaluate_sample 3 \
    --n_select_sample 5 \
    --task_start_index {START} \
    --task_end_index {END} \
    --temperature 0.7 \
    --verbose